# **Fitur Unik 2: Penjelasan Prediksi dengan Integrated Gradients**

Model final memberi satu angka probabilitas risiko diabetes. Untuk aplikasi kesehatan, angka saja tidak cukup. Pengguna perlu tahu **faktor apa yang membuat risikonya tinggi atau rendah**, dan tim pengembang perlu memastikan model mengambil keputusan berdasarkan pola yang masuk akal secara medis, bukan pola yang kebetulan.

Fitur unik kedua menjawab kebutuhan ini dengan **Integrated Gradients** (IG), metode atribusi untuk jaringan saraf dari Sundararajan dkk. (2017). IG menghitung kontribusi setiap fitur terhadap selisih probabilitas antara responden yang dijelaskan dan sebuah **baseline**. IG ditulis sendiri dengan `tf.GradientTape` di `src/inference.py`, jadi aplikasi memakai kode yang sama persis.

Supaya fitur ini tidak sekadar tampilan, dampaknya dibuktikan lewat empat pengujian di data validation:

| Pengujian | Pertanyaan yang dijawab |
|---|---|
| Completeness | Apakah jumlah kontribusi sama dengan selisih probabilitas |
| Deletion test | Apakah fitur yang ditunjuk IG benar-benar yang paling menggerakkan prediksi |
| Sensitivitas baseline | Apakah kesimpulan berubah kalau baseline diganti |
| Kesesuaian domain | Apakah arah pengaruh fitur sesuai dengan faktor risiko diabetes yang dikenal |

Model yang dijelaskan adalah model final (`models/mlp_final.keras`). Data test tetap tidak disentuh.

**Penanggung jawab:** Erik Hamli (231110980)

## **1. Import Library dan Konfigurasi**

Notebook ini tidak melatih model, jadi tidak ada saklar `RUN_EXPERIMENTS`. Seluruh perhitungan selesai dalam beberapa menit.

In [2]:
import sys
import json
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from scipy.stats import spearmanr

from src import config
from src import preprocessing as prep
from src import evaluation as ev
from src import inference as inf
from src import visualization as viz

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
viz.set_style()
WARNA_NAIK, WARNA_TURUN, WARNA_UTAMA = "#eb6834", "#2a78d6", "#2a78d6"
rng = np.random.default_rng(config.SEED)
print("TensorFlow:", tf.__version__)

TensorFlow: 2.21.0


## **2. Memuat Data dan Model Final**

Probabilitas model final di data validation dibandingkan dengan probabilitas yang tersimpan di notebook 05, untuk memastikan model yang dijelaskan adalah model yang sama.

In [3]:
data = prep.prepare_datasets()
assert data["test_fingerprint"] == prep.load_preprocessing_config()["split"]["test_fingerprint"]
model_cfg = json.loads((config.MODELS_DIR / "model_config.json").read_text(encoding="utf-8"))
model = keras.models.load_model(ROOT / model_cfg["model_path"], compile=False)
THRESHOLD = model_cfg["threshold"]

X_val, y_val = data["X_val"], data["y_val"].astype(int)
X_val_raw = data["X_val_raw"].reset_index(drop=True)
prob_val = inf.predict_proba(model, X_val)
prob_05 = np.load(config.RESULTS_DIR / "05_prob_validation_seed42.npz")["final"]

print(f"Model      : {model_cfg['model_path']} | loss {model_cfg['loss']} | threshold {THRESHOLD}")
print(f"Validation : {X_val.shape}")
print("Probabilitas sama dengan notebook 05:", np.allclose(prob_val, prob_05, atol=1e-6))

Model      : models/mlp_final.keras | loss BCE | threshold 0.13
Validation : (34332, 21)
Probabilitas sama dengan notebook 05: True
